
<h3 align="center">SA4110 MACHINE LEARNING APPLICATION DEVELOPMENT</h3>
<h4 align="center">CA - TEAM PROJECT - IMAGE CLASSIFIER</h4>
<hr>

<div class="alert alert-block alert-info">
<b><u>Tasks:</u></b>
<ol>
<li>Create an image classifier (CNN model) to classify images of fruits correctly.</li>
<li>A "Fruits" dataset is provided that consists of these 3 Classes: -</li>
    <ul>
    <li>Apple</li>
    <li>Banana</li>
    <li>Orange</li>
    </ul>
<li>Use the images in train.zip and test.zip to train and test the image classifier.</li>
<li>Document experiments and results in improving model’s accuracy.</li>
<li>The following activities can improve model’s accuracy: -</li>
    <ul>
    <li>Balance out the number of samples in each class</li>
    <li>Correct any mis-labelling in any of the 3 classes</li>
    <li>Image augmentation to generate more data </li>
    </ul>
<li>Use Matplotlib to generate any plots</li>
</ol></div>



In [1]:
!pip -q install "keras>=3.0" torch torchvision seaborn scikit-learn
print("dependencies ready")

dependencies ready


In [ ]:
from google.colab import files
from pathlib import Path
import os

os.makedirs("/content/fruits/train", exist_ok=True)
os.makedirs("/content/fruits/test", exist_ok=True)

uploaded = files.upload()
for fname in uploaded:
    with open(f"/content/fruits/train/{fname}", "wb") as f:
        f.write(uploaded[fname])

uploaded = files.upload()
for fname in uploaded:
    with open(f"/content/fruits/test/{fname}", "wb") as f:
        f.write(uploaded[fname])

DATA_ROOT = Path("/content/fruits")

In [ ]:
import os
import statistics
from pathlib import Path
os.environ["KERAS_BACKEND"] = "torch"

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML
from PIL import Image
import torch
import keras
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
### Load the Testing Datasets

# Define the Path Directory to 'test' directory
img_test_dir = DATA_ROOT / "test"

""" Generate Classification Labels associated with the Respective Fruit Classes based on FileName """

# Initialize the Required Parameters for the Classification Label
img_t_fileNames = []
img_t_classLabel = []

for img_path in img_test_dir.glob("*.jpg"):
    try:
        img_t_fileNames.append(img_path.name)
        img_t_classLabel.append(img_path.name.strip().split('_')[0])
    except Exception as e:
        print(f"Unable to Read {img_path.name}: {e}")

class_test_df = pd.DataFrame({
    'fileName' : img_t_fileNames,
    'classLabel' : img_t_classLabel
})

display(HTML("<h3>Display (10) Random Sample of the Class Label DataFrame:</h3>"))
display(class_test_df.sample(10)
        .style
        .highlight_null(color='lightcoral')
        .set_properties(**{'color':'white'})
        .set_table_styles([
            {'selector':'tr:nth-child(even)', 'props': [('background-color', '#708090')]},
            {'selector':'th', 'props':[('text-align', 'center')]},
            {'selector':'tr:nth', 'props':[('text-align', 'center')]}
        ]))
print("-" * 50)

for i, label in enumerate(class_test_df['classLabel'].unique(), start = 1):
    print(f"Class Label {i}: {label}")
print("-" * 50)

# Calculting the Images Median Width and Height to Determine the Target Size
img_t_widths = []
img_t_heights = []

for img_path in img_test_dir.glob("*.jpg"):
    try:
        with Image.open(img_path) as img:
            img_t_widths.append(img.size[0])
            img_t_heights.append(img.size[1])
    except Exception as e:
        print(f"Unable to Read {img_path.name}: {e}")

if img_t_widths and img_t_heights:
    img_t_widths_median = statistics.median(img_t_widths)
    img_t_heights_median = statistics.median(img_t_heights)

    print(f"Total No. of Images Evaluated: {len(img_t_widths)}")
    print(f"Image Median Widths: {img_t_widths_median}")
    print(f"Image Median Heights: {img_t_heights_median}")
else:
    print(f"No .jpg Images found in the Specified Directory.")

In [ ]:
### Define a New Class: FruitLabeller

class FruitLabeller(Dataset):
    def __init__(self, class_dataframe, img_directory, transforms = None):
        self.dataframe = class_dataframe
        self.directory = img_directory
        self.transforms = transforms

        unique_labels = sorted(class_dataframe['classLabel'].unique())
        self.label_map = {label: idx for idx, label in enumerate(unique_labels)}

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_name = self.dataframe.iloc[idx]['fileName']
        img_path = self.directory / img_name
        image = Image.open(img_path).convert("RGB")

        label_str = self.dataframe.iloc[idx]['classLabel']
        label = self.label_map[label_str]

        if self.transforms:
            image = self.transforms(image)

        return image, label

def display_dataset_shape(df, dataset_name):
    shape_df = df['classLabel'].value_counts().sort_index().reset_index()
    shape_df.columns = ['Class Label', 'Total Image']

    display(HTML(f"<h3>{dataset_name} Dataset Shape:</h3>"))
    display(shape_df.style
            .set_properties(**{'color':'white', 'text-align':'center'})
            .set_table_styles([
            {'selector':'tr:nth-child(even)', 'props': [('background-color', '#708090')]},
            {'selector':'th', 'props':[('text-align', 'center')]},
            ])
            .hide(axis="index")
            )

In [ ]:
### Loading the Image Data in PyTorch

# Set Target Size (Max 224x224)
target_width = int(img_T_widths_median) if img_T_widths_median < 224 else 224
target_height = int(img_T_heights_median) if img_T_heights_median < 224 else 224

# Split Data (80% Train, 20% Val)
train_df, val_df = train_test_split(
    class_train_df,
    test_size=0.2,
    random_state=42,
    stratify=class_train_df['classLabel']
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

# Define PyTorch Transformations
train_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=60),
    transforms.ColorJitter(brightness=0.2),
    transforms.RandomCrop((target_height, target_width)),
    transforms.ToTensor()
])

test_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop((target_height, target_width)),
    transforms.ToTensor()
])

# Instantiate PyTorch Datasets
train_dataset = FruitLabeller(train_df, img_train_dir, train_transforms)
val_dataset = FruitLabeller(val_df, img_train_dir, test_transforms)
test_dataset = FruitLabeller(class_test_df, img_test_dir, test_transforms)

# Create PyTorch DataLoaders
train_loader = DataLoader(train_dataset, batch_size=9, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=9, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=9, shuffle=False)

# Display Dataset Shapes
display_dataset_shape(train_df, "Training (80%)")
display_dataset_shape(val_df, "Validation (20%)")
display_dataset_shape(class_test_df, "Testing")

In [ ]:
### Display Sample Images

images, labels = next(iter(train_loader))
class_names = {v: k for k, v in train_dataset.label_map.items()}

# Set up a 5 x 5 MatPlotLib Grid
fig, axes = plt.subplots(3, 3, figsize=(10, 10))

for i, ax in enumerate(axes.flat):
    img = np.transpose(images[i].numpy(), (1, 2, 0))
    img = np.clip(img, 0, 1)
    ax.imshow(img)

    # Look up the Integer Label in the Dictionary and Set it as the Title
    label_idx = labels[i].item()
    ax.set_title(class_names[label_idx].capitalize(), fontsize=12, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
### Establishing the CNN Architecture
# Input Layer   -> (224 x 224 x 3)

model = keras.Sequential([
    keras.layers.Input(shape=(3, 224, 224)),

    # 32 filters, Output shape: 224 x 224 x 32
    keras.layers.Conv2D(
        filters=32,
        kernel_size=(3, 3),
        padding="same",
        activation="relu",
        data_format="channels_first"
    ),

    # MaxPool: 224 x 224 -> 112 x 112
    keras.layers.MaxPooling2D(pool_size=(2, 2), data_format="channels_first"),

    # 64 filters, output shape: 112 x 112 x 64
    keras.layers.Conv2D(
        filters=64,
        kernel_size=(3, 3),
        padding="same",
        activation="relu",
        data_format="channels_first"
    ),

    # MaxPool: 112 x 112 -> 56 x 56
    keras.layers.MaxPooling2D(pool_size=(2, 2), data_format="channels_first"),


    # 128 filters, output shape: 56 x 56 x 128
    keras.layers.Conv2D(
        filters=128,
        kernel_size=(3, 3),
        padding="same",
        activation="relu",
        data_format="channels_first"
    ),

    # MaxPool: 56 x 56 -> 28 x 28
    keras.layers.MaxPooling2D(pool_size=(2, 2), data_format="channels_first"),

    # Flatten: 28 x 28 x 128 = 100,352
    keras.layers.Flatten(),

    # Fully connected layer
    keras.layers.Dense(128, activation="relu"),

    # Dropout
    keras.layers.Dropout(0.5),

    # Output Layer: 3 nodes (Apples, Bananas, Oranges)
    keras.layers.Dense(3, activation='softmax')
])

# Display the network structure and parameter count
model.summary()

In [ ]:
### Compile the Model
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

In [ ]:
### Train the Model
history = model.fit(
    train_loader,
    validation_data=val_loader,
    epochs=20
)

In [ ]:
### Visualize Training History

# Extract Key Metrics from the History object
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs_range = range(1, len(acc) + 1)

sns.set_theme(style='darkgrid')
plt.figure(figsize=(12, 5))

# Plot Accuracy
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy', marker='o', color='blue')
plt.plot(epochs_range, val_acc, label='Validation Accuracy', marker='o', color='darkorange')
plt.title('Training and Validation Accuracy', fontsize=12, fontweight='bold')
plt.xlabel('Epochs', fontsize=10, fontweight='bold')
plt.ylabel('Accuracy', fontsize=10, fontweight='bold')
plt.legend(loc='lower right')
plt.xlim(0,21)
plt.ylim(0,1.1)
plt.grid(True)

# Plot Loss
plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss', marker='o', color='blue')
plt.plot(epochs_range, val_loss, label='Validation Loss', marker='o', color='darkorange')
plt.title('Training and Validation Loss', fontsize=12, fontweight='bold')
plt.xlabel('Epochs', fontsize=10, fontweight='bold')
plt.ylabel('Loss', fontsize=10, fontweight='bold')
plt.legend(loc='upper right')
plt.xlim(0,21)
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
### Evaluate the Model on Test Data
test_loss, test_accuracy = model.evaluate(test_loader)
print(f"{'Final Test Loss':<20} {': '} {test_loss:.4f}")
print(f"{'Final Test Accuracy':<20} {': '} {test_accuracy:.4f}")

In [ ]:
### Comprehensive Test Evaluation

# Initialize lists to store true and predicted labels for the entire dataset
all_true_labels = []
all_pred_labels = []

# Iterate through the entire test_loader
for images, labels in test_loader:
    # Get predictions for the current batch (verbose=0 suppresses per-batch output)
    batch_preds_prob = model.predict(images, verbose=0)
    batch_preds = np.argmax(batch_preds_prob, axis=1)

    # Append the results to the tracking lists
    all_true_labels.extend(labels.numpy())
    all_pred_labels.extend(batch_preds)

# Generate the Classification Report
# Extract class names in the correct index order
class_names_list = [name.capitalize() for name in train_dataset.label_map.keys()]

print("\n" + "="*50)
print("Classification Report")
print("="*50)
print(classification_report(all_true_labels, all_pred_labels, target_names=class_names_list))

# Generate and Plot the Confusion Matrix
cm = confusion_matrix(all_true_labels, all_pred_labels)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names_list,
            yticklabels=class_names_list)
plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Label', fontsize=12, fontweight='bold')
plt.ylabel('True Label', fontsize=12, fontweight='bold')
plt.show()